In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval
except Exception:
    pass

In [2]:
import json
import warnings
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
import polars as pl

from dataeval import Metadata
from dataeval.bias import Balance

In [3]:


def collection(seed: int, mean_temp: float, n: int = 600):
    rng = np.random.default_rng(seed)
    temp = rng.normal(mean_temp, 9.0, n)
    factors = {
        "temp_c": temp,
        "haze": np.clip(0.55 * (temp - mean_temp) / 9.0 + rng.normal(0.0, 0.9, n), -3, 3),
        "site": np.array(["ridge", "valley", "coast"])[rng.integers(0, 3, n)],
    }
    return factors, rng.integers(0, 4, n)


winter_factors, winter_labels = collection(seed=0, mean_temp=-2.0)
spring_factors, spring_labels = collection(seed=1, mean_temp=14.0)

In [4]:
winter = Metadata.from_factors(winter_factors, class_labels=winter_labels)
spring = Metadata.from_factors(spring_factors, class_labels=spring_labels)

# Binning is lazy, so touch both now: the warning below is the subject of this guide
# and is worth reading once, here, rather than interleaved through every later cell.
_ = winter.factor_data
_ = spring.factor_data

for name, spec in winter.encoding().items():
    print(f"{name:8} {type(spec).__name__:10} provenance={spec.provenance}")

haze     BinSpec    provenance=derived
site     LevelSpec  provenance=derived
temp_c   BinSpec    provenance=derived


/tmp/ipykernel_4758/2245003410.py:6: UserWarning: `haze` and `temp_c` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"haze": [...]} to control this.
  _ = winter.factor_data
/tmp/ipykernel_4758/2245003410.py:7: UserWarning: `haze` and `temp_c` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"haze": [...]} to control this.
  _ = spring.factor_data


In [5]:
print(f"winter temp_c: {np.round(winter.encoding('temp_c').edges, 1)}")
print(f"spring temp_c: {np.round(spring.encoding('temp_c').edges, 1)}")

winter temp_c: [ -inf -21.4  -5.8   9.9   inf]
spring temp_c: [-inf -2.6  8.5 19.7 30.8  inf]


In [6]:
# `haze` stays automatic for the rest of this guide and would repeat that advice at
# every construction below, so it is silenced after the first reading.
warnings.filterwarnings("ignore", message=".*binned automatically")

In [7]:
FREEZING = [-np.inf, 0.0, 10.0, np.inf]  # below freezing / cold / mild
SITES = ["ridge", "valley", "coast"]

winter = Metadata.from_factors(
    winter_factors,
    class_labels=winter_labels,
    continuous_factor_bins={"temp_c": FREEZING},
    factor_levels={"site": SITES},
)

for name, spec in winter.encoding().items():
    print(f"{name:8} {type(spec).__name__:10} provenance={spec.provenance}")

haze     BinSpec    provenance=derived
site     LevelSpec  provenance=declared
temp_c   BinSpec    provenance=edges


In [8]:
try:
    incomplete = Metadata.from_factors(
        winter_factors,
        class_labels=winter_labels,
        factor_levels={"site": ["ridge", "valley"]},
        strict=True,
    )
    _ = incomplete.factor_data
except ValueError as error:
    print(error)

Values ['coast'] are not in this factor's declared vocabulary ['ridge', 'valley']. Pass strict=False to admit them, or widen the declaration.


In [9]:
before = winter.factor_data.copy()
winter.accept()
print(f"provenance: { ({n: s.provenance for n, s in winter.encoding().items()}) }")
print(f"codes unchanged: {np.array_equal(before, winter.factor_data)}")

provenance: {'haze': 'accepted', 'site': 'declared', 'temp_c': 'edges'}
codes unchanged: True


In [10]:
workspace = Path(mkdtemp(prefix="dataeval-encoding-"))
descriptor = workspace / "encoding.json"
winter.export_encoding(descriptor)

print(json.dumps(json.loads(descriptor.read_text())["factors"]["temp_c"], indent=2))

{
  "edges": [
    "-inf",
    0.0,
    10.0,
    "inf"
  ],
  "kind": "bins",
  "method": null,
  "provenance": "edges"
}


In [11]:
spring = Metadata.from_factors(spring_factors, class_labels=spring_labels, encoding=descriptor)

print(f"winter temp_c: {winter.encoding('temp_c').edges}")
print(f"spring temp_c: {spring.encoding('temp_c').edges}")

winter temp_c: (-inf, 0.0, 10.0, inf)
spring temp_c: (-inf, 0.0, 10.0, inf)


In [12]:
print(f"winter: {winter.encoding_digest}")
print(f"spring: {spring.encoding_digest}")
print(f"comparable: {winter.encoding_digest == spring.encoding_digest}")

result = Balance().evaluate(spring)
print(f"on the result: {result.meta().state['encoding_digest']}")

winter: 619615eaeb7391db
spring: 619615eaeb7391db
comparable: True
on the result: 619615eaeb7391db


In [13]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    stale = Metadata.from_factors(
        spring_factors,
        class_labels=spring_labels,
        continuous_factor_bins={"temp_c": [-np.inf, -20.0, -10.0, 0.0, np.inf]},
    )
    _ = stale.factor_data

for warning in caught:
    if "bins unused" in str(warning.message):
        print(str(warning.message))

Declared cuts left bins unused: temp_c (3 of 4 bins hold rows). The encoding still applies and the codes are unchanged; this is the data no longer matching the policy, not an error. Call reencode() to refit — which moves codes — or leave it and read the empty groups as the finding they are.


In [14]:
mixed = Metadata.from_factors(
    winter_factors,
    class_labels=winter_labels,
    continuous_factor_bins={"temp_c": FREEZING},
    factor_levels={"site": SITES},
)

print(Balance().evaluate(mixed).factors.filter(pl.col("factor1") == "temp_c"))

shape: (2, 5)
┌─────────┬─────────┬──────────┬───────────────┬───────────┐
│ factor1 ┆ factor2 ┆ mi_value ┆ is_correlated ┆ scored_as │
│ ---     ┆ ---     ┆ ---      ┆ ---           ┆ ---       │
│ cat     ┆ cat     ┆ f64      ┆ bool          ┆ cat       │
╞═════════╪═════════╪══════════╪═══════════════╪═══════════╡
│ temp_c  ┆ haze    ┆ 0.211675 ┆ false         ┆ estimator │
│ temp_c  ┆ site    ┆ 0.001469 ┆ false         ┆ table     │
└─────────┴─────────┴──────────┴───────────────┴───────────┘


In [15]:
for source in ("auto", "coded", "values"):
    factors = Balance(factor_source=source).evaluate(mixed).factors
    row = factors.filter((pl.col("factor1") == "temp_c") & (pl.col("factor2") == "haze"))
    print(f"{source:8} mi_value={row['mi_value'][0]:.4f}  scored_as={row['scored_as'][0]}")

auto     mi_value=0.2117  scored_as=estimator
coded    mi_value=0.2357  scored_as=linfoot


values   mi_value=0.3489  scored_as=estimator


In [16]:
archive = workspace / "winter.dem"
winter.save(archive)
restored = Metadata.load(archive)

print(f"edges:  {restored.encoding('temp_c').edges}")
print(f"digest: {restored.encoding_digest == winter.encoding_digest}")

edges:  (-inf, 0.0, 10.0, inf)
digest: True
